# 06 — Selección de Características
## Pipeline CAD-TBX11K | Diagnóstico Asistido por Computador en Tuberculosis

Este notebook selecciona las características usando **todas las imágenes**
(8256 tras eliminar duplicados exactos). **No hay hold-out**: la evaluación se hace
con validación cruzada sobre las 8256 y la selección de características se ajusta
**dentro de cada fold**.

### Objetivo
Elegir el método de selección (Chi², Mutual Information o importancia de Random Forest)
y el número de características `N`, maximizando el F1-macro en validación cruzada.

### Protocolo
1. Carga de las 8256 imágenes canónicas (`df_model.csv`) y verificación de duplicados.
2. **CV 5-fold estratificada sobre las 8256**.
3. En cada fold, el selector se ajusta **solo con el fold de entrenamiento**.
4. Regla de **1 error estándar** para elegir el menor `N` dentro de 1 SE del mejor.
5. Persistencia de `X_full.npy`, `y_full.npy`, la selección y los scores.

### Flujo

```
features_tbx11k.csv (8256 x 1860)
         |
         v
 Verificacion de duplicados -> 8256 imagenes
         |
         v
 CV 5-fold: selector ajustado dentro de cada fold
 (Chi2 / MI / RF) x N en {100..1860}
         |
         v
 Regla 1-SE (error estandar) -> metodo + N
         |
         v
 X_full.npy / y_full.npy + seleccion
         |
         v
     Notebook 07
  Desbalanceo (CV sobre 8256)
```


## 0. Configuración del entorno


In [ ]:
# Se configura el path del proyecto y se importan las librerías necesarias.
# Referencias:
#   validación cruzada: https://scikit-learn.org/stable/modules/cross_validation.html
#   selección de características: https://scikit-learn.org/stable/modules/feature_selection.html

import sys, os, warnings, json, time, hashlib
import cv2
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

_CWD = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(_CWD, '..')) if os.path.basename(_CWD) == 'notebooks' else _CWD
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.features import model_feature_columns
from src.selection import feature_scores
from src.visualization import set_publication_style

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path(PROJECT_ROOT) / 'data'

set_publication_style()
print('Entorno configurado correctamente.')


## 1. Carga y deduplicación exacta


In [ ]:
# Se carga la matriz canonica de 8.256 imagenes. La deduplicacion exacta ya se
# hizo en df_model (src/dedup_drop_names.csv); aqui solo se verifican duplicados.
# No se crea ningun split train/test: la estimacion es por validacion cruzada.

CSV_PATH = DATA_DIR / 'features_tbx11k.csv'
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f'No se encontro {CSV_PATH.resolve()}. Ejecuta primero el notebook 05.'
    )

df = pd.read_csv(CSV_PATH)
fallback = df['mask_fallback'].astype(bool)
print(f'Fallback de segmentacion: {int(fallback.sum())} ({fallback.mean():.1%})')

# Cambiar a True si se quiere entrenar solo con mascaras que superaron QC.
EXCLUIR_FALLBACK = False
if EXCLUIR_FALLBACK:
    df = df[~fallback].reset_index(drop=True)
    print(f'Imagenes tras excluir fallback: {len(df)}')
META_COLS = ['image_path', 'class_name', 'label']
feat_cols = model_feature_columns(df)

# Verificacion: un mismo vector de features no debe tener etiquetas distintas.
claves = pd.util.hash_pandas_object(df[feat_cols], index=False)
conflictos = (
    df.assign(_clave=claves)
      .groupby('_clave')['label']
      .nunique()
      .gt(1)
      .sum()
)
n_dup = int(pd.Series(claves).duplicated().sum())
print(f'Vectores de features repetidos: {n_dup}')
print(f'Vectores repetidos con etiquetas en conflicto: {conflictos}')
print(f'Imagenes: {len(df)}')
print(df['class_name'].value_counts().to_string())

# Verificacion complementaria: duplicados exactos a nivel de archivo de imagen.
def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if df['image_path'].map(os.path.exists).all():
    hashes_img = df['image_path'].map(_md5)
    print(f'Duplicados exactos a nivel de imagen: {int(hashes_img.duplicated().sum())}')
else:
    print('Aviso: algunas rutas no existen; no se verifico el hash de imagen.')

# Verificacion perceptual: dHash 8x8 (duplicados visuales exactos).
def _dhash(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (9, 8), interpolation=cv2.INTER_AREA)
    diff = img[:, 1:] > img[:, :-1]
    return int(''.join('1' if x else '0' for x in diff.flatten()), 2)

if df['image_path'].map(os.path.exists).all():
    dhashes = df['image_path'].map(_dhash)
    print(f'Duplicados perceptuales exactos (dHash): {int(dhashes.duplicated().sum())}')

# La matriz canonica se conserva completa: 8.256 imagenes.
X = df[feat_cols].to_numpy(dtype=np.float32)
y = df['label'].to_numpy()
meta = df[META_COLS].copy()

np.save(DATA_DIR / 'X_full.npy', X)
np.save(DATA_DIR / 'y_full.npy', y)
meta.to_csv(DATA_DIR / 'full_meta.csv', index=False)

print()
print(f'X_full : {X.shape} ({X.dtype})')
print(f'y_full : {y.shape}')
print(f'Features: {len(feat_cols)}')


## 2. Redundancia interna de HOG (medida, no supuesta)

Se mide la correlación entre una muestra de features HOG para documentar su
redundancia interna. No se incluye PHOG: el modelo usa textura + HOG.


In [ ]:
# Se mide la correlación entre una muestra de features HOG.
# Referencia: https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html

hog_cols = [c for c in feat_cols if c.startswith('hog_')]
rng = np.random.RandomState(RANDOM_STATE)
n_rows = min(2000, len(df))
n_cols = min(100, len(hog_cols))

rows = rng.choice(len(df), size=n_rows, replace=False)
hog_sample = rng.choice(len(hog_cols), size=n_cols, replace=False)
cols = [hog_cols[i] for i in hog_sample]

M = df.iloc[rows][cols].to_numpy(dtype=np.float32)
corr = np.corrcoef(M, rowvar=False)
within_hog = np.abs(corr[np.triu_indices(n_cols, 1)])

redundancia = {
    'n_filas_muestra': int(n_rows),
    'n_hog_muestra': int(n_cols),
    'corr_abs_media_hog': float(np.nanmean(within_hog)),
}
print(json.dumps(redundancia, indent=2))
with open(DATA_DIR / '06_redundancia_hog.json', 'w', encoding='utf-8') as f:
    json.dump(redundancia, f, ensure_ascii=False, indent=2)


## 3. Validación cruzada de la selección de características


In [ ]:
# Para cada fold se calculan los scores de los tres métodos usando SOLO el train
# del fold; después se evalúa cada N con un LogisticRegression(class_weight='balanced').
# El selector nunca ve el fold de validación.
# Referencia: https://scikit-learn.org/stable/modules/feature_selection.html

N_VALUES = [100, 300, 500, 1000, 1500, 2000, 2200]
CV_FOLDS = 5
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

filas = []
rankings = {m: {} for m in ['Chi2', 'MI', 'RF']}
t0 = time.time()

for fold, (tr, va) in enumerate(cv.split(X, y), start=1):
    X_tr, X_va = X[tr], X[va]
    y_tr, y_va = y[tr], y[va]

    scores = {
        'Chi2': feature_scores('Chi2', X_tr, y_tr, RANDOM_STATE),
        'MI':   feature_scores('MI', X_tr, y_tr, RANDOM_STATE),
        'RF':   feature_scores('RF', X_tr, y_tr, RANDOM_STATE),
    }

    for method, sc in scores.items():
        orden = np.argsort(sc)[::-1]
        rankings[method][fold] = orden
        for n in N_VALUES:
            if n > len(orden):
                continue
            idx = orden[:n]
            scaler = StandardScaler()
            Xtr_s = scaler.fit_transform(X_tr[:, idx])
            Xva_s = scaler.transform(X_va[:, idx])
            clf = LogisticRegression(
                class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE,
            )
            clf.fit(Xtr_s, y_tr)
            pred = clf.predict(Xva_s)
            filas.append({
                'method': method,
                'n': n,
                'fold': fold,
                'f1_macro': f1_score(y_va, pred, average='macro'),
            })

    print(f'fold {fold}/{CV_FOLDS} listo ({time.time() - t0:.1f}s)')

resultados_fold = pd.DataFrame(filas)
resumen = (
    resultados_fold
    .groupby(['method', 'n'])['f1_macro']
    .agg(['mean', 'std'])
    .reset_index()
)
resumen['se'] = resumen['std'] / np.sqrt(CV_FOLDS)
resumen = resumen.sort_values(['mean', 'n'], ascending=[False, True]).reset_index(drop=True)

print()
print('Resultados (media, DE y error estándar):')
print(resumen.round(6).to_string(index=False))

resumen.to_csv(DATA_DIR / '06_cv_resultados.csv', index=False)
resultados_fold.to_csv(DATA_DIR / '06_cv_por_fold.csv', index=False)


## 4. Selección por regla de 1 error estándar


In [ ]:
# Regla de 1 error estándar (1-SE): se toma el mejor F1-macro medio y se aceptan
# configuraciones dentro de 1 error estándar (std / sqrt(k)) del mejor.
# Entre ellas se elige el menor N y, a igual N, el mayor F1.
# Referencia: Breiman et al. (1984), regla de 1-SE.

best = resumen.loc[resumen['mean'].idxmax()]
umbral = best['mean'] - best['se']

candidatos = resumen[resumen['mean'] >= umbral].copy()
candidatos = candidatos.sort_values(['n', 'mean'], ascending=[True, False])
sel = candidatos.iloc[0]

METHOD_SEL = sel['method']
N_SEL = int(sel['n'])

# Estabilidad de la selección entre folds (Jaccard del top-N).
top_sets = [set(rankings[METHOD_SEL][f][:N_SEL]) for f in range(1, CV_FOLDS + 1)]
jaccard = []
for i in range(len(top_sets)):
    for j in range(i + 1, len(top_sets)):
        jaccard.append(len(top_sets[i] & top_sets[j]) / len(top_sets[i] | top_sets[j]))
estabilidad = float(np.mean(jaccard))
print(f'Estabilidad (Jaccard medio top-{N_SEL} entre folds): {estabilidad:.4f}')

print('Regla de 1 error estándar:')
print(f'  Mejor configuración : {best["method"]}, N={int(best["n"])} (F1={best["mean"]:.6f})')
print(f'  Error estándar      : {best["se"]:.6f}')
print(f'  Umbral              : {umbral:.6f}')
print()
print(f'Configuraciones candidatas: {len(candidatos)}')
print(candidatos.round(6).to_string(index=False))
print()
print(f'Seleccionada (menor N dentro de 1 SE): {METHOD_SEL}, N={N_SEL}')
print(f'  F1-macro CV : {sel["mean"]:.6f} +/- {sel["std"]:.6f}')

seleccion = {
    'criterio': '1-SE con error estandar',
    'n_folds': CV_FOLDS,
    'metodo': METHOD_SEL,
    'n_features': N_SEL,
    'mejor_metodo': best['method'],
    'mejor_n': int(best['n']),
    'mejor_f1': float(best['mean']),
    'error_estandar': float(best['se']),
    'umbral': float(umbral),
    'estabilidad_jaccard_media': estabilidad,
}
with open(DATA_DIR / '06_seleccion.json', 'w', encoding='utf-8') as f:
    json.dump(seleccion, f, ensure_ascii=False, indent=2)

# Scores definitivos sobre TODAS las imágenes (solo para guardar nombres/scores;
# el modelo final volverá a ajustar el selector dentro de su propio pipeline).
scores_all = feature_scores(METHOD_SEL, X, y, RANDOM_STATE)
orden_all = np.argsort(scores_all)[::-1]
nombres_sel = [feat_cols[i] for i in orden_all[:N_SEL]]
with open(DATA_DIR / '06_selected_feature_names.json', 'w', encoding='utf-8') as f:
    json.dump(nombres_sel, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    'feature': [feat_cols[i] for i in orden_all],
    'score': np.sort(scores_all)[::-1],
}).to_csv(DATA_DIR / '06_scores.csv', index=False)

# Curva F1 vs N.
fig, ax = plt.subplots(figsize=(8, 4.5))
for method in ['Chi2', 'MI', 'RF']:
    sub = resumen[resumen['method'] == method].sort_values('n')
    ax.errorbar(sub['n'], sub['mean'], yerr=sub['std'], marker='o',
                capsize=3, lw=1.4, label=method)
ax.axvline(N_SEL, color='k', linestyle='--', lw=0.8,
           label=f'Seleccionado: N={N_SEL}')
ax.set_xlabel('Número de características (N)', fontsize=11)
ax.set_ylabel('F1-macro (CV 5-fold)', fontsize=11)
ax.set_title('Selección de características — todas las imágenes', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
fig.tight_layout()
plt.show()


In [ ]:
# Composicion de la seleccion por familia: evita que HOG domine sin control.
from src.features import get_feature_groups

grupos = get_feature_groups(df)
seleccionadas = set(nombres_sel)
composicion = []
for familia, columnas in grupos.items():
    n = sum(c in seleccionadas for c in columnas)
    if n:
        composicion.append({
            'familia': familia,
            'n_seleccionadas': n,
            'n_disponibles': len(columnas),
            'fraccion_familia': n / len(columnas),
        })
pd.DataFrame(composicion).sort_values('n_seleccionadas', ascending=False).round(3)


In [ ]:
# Diagnostico: textura sola vs HOG solo. Si HOG domina demasiado, el modelo
# puede estar aprendiendo geometria del torax en lugar de senal pulmonar.
from src.features import get_feature_groups
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

grupos_diag = get_feature_groups(df)
y_diag = df['label'].to_numpy()
cv_diag = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
comparacion_familias = []
for familia in ['texture', 'hog']:
    cols_familia = grupos_diag[familia]
    Xf = df[cols_familia].to_numpy(dtype=np.float32)
    f1_folds = []
    for tr, va in cv_diag.split(Xf, y_diag):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xf[tr])
        Xva = scaler.transform(Xf[va])
        clf = LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
        )
        clf.fit(Xtr, y_diag[tr])
        f1_folds.append(f1_score(y_diag[va], clf.predict(Xva), average='macro'))
    comparacion_familias.append({
        'familia': familia,
        'n_features': len(cols_familia),
        'f1_macro_media': float(np.mean(f1_folds)),
        'f1_macro_std': float(np.std(f1_folds)),
    })
pd.DataFrame(comparacion_familias).round(4)


## Conclusiones

Se usaron **todas las imágenes** (8256 tras deduplicar). No hay hold-out ni conjunto de
test separado: la estimación se obtiene por **validación cruzada 5-fold sobre las 8256**.

En cada fold, el selector de características se ajustó **solo con el fold de entrenamiento**.
Se compararon Chi², Mutual Information e importancia de Random Forest sobre un rango de N,
y se eligió el menor N dentro de **1 error estándar** del mejor F1-macro.

Los artefactos `X_full.npy` y `y_full.npy` contienen las 8256 imágenes; el notebook 07
usará exactamente esos datos para la selección de la estrategia de desbalanceo.

**Limitación documentada:** `N` se elige por CV sobre las 8256 y se reutiliza en el
notebook 08; no se retunea dentro de cada fold del 08 por coste computacional.
La estabilidad entre folds se reporta con el Jaccard del top-N.

Se verificaron duplicados exactos por hash de archivo y duplicados perceptuales exactos
con dHash. dHash detecta imágenes idénticas o casi idénticas, pero no recortes o rotaciones;
esa comprobación queda como limitación.


**Siguiente etapa:** `07_Desbalanceo.ipynb`
